<a href="https://colab.research.google.com/github/Syifamaulvi/InformationBaseLearning/blob/main/Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Algorithm 1 Pseudocode description of the ID3 algorithm.**

Require: set of descriptive features d  
Require: set of training instances D  

1: if all the instances in D have the same target level C then return a decision tree consisting of a leaf node with label C  

2: else if d is empty then return a decision tree consisting of a leaf node with the label of the majority target level in D  

3: else if D is empty then return a decision tree consisting of a leaf node with the label of the majority target level of the dataset of the immediate parent node  

4: else  

5:     d[best] ← arg max_d IG(d, D)  

6:     make a new node, Node[d_best], and label it with d[best]  

7:     partition D using d[best]  

8:     remove d[best] from d  

9:     for each partition Di of D do  

10:         grow a branch from Node[d_best] to the decision tree created by rerunning ID3 with D = Di  

11:     end for  

12: end if  

# **Studi Kasus**

Sebuah taman bermain ingin memutuskan,
Apakah anak-anak jadi bermain di luar atau tidak?

Keputusannya dipengaruhi oleh 3 atribut:

**Cuaca:** Cerah, Berawan, Hujan

**Suhu:** Panas, Sejuk, Dingin

**Kelembapan:** Tinggi, Normal

dataset:

| No | Cuaca   | Suhu   | Kelembapan | Bermain |
| -- | ------- | ------ | ---------- | ------- |
| 1  | Cerah   | Panas  | Tinggi     | Tidak   |
| 2  | Cerah   | Panas  | Normal     | Tidak   |
| 3  | Berawan | Panas  | Tinggi     | Ya      |
| 4  | Hujan   | Sejuk  | Tinggi     | Ya      |
| 5  | Hujan   | Dingin | Normal     | Ya      |
| 6  | Hujan   | Dingin | Tinggi     | Tidak   |


In [ ]:
import math
from collections import Counter, defaultdict

In [ ]:
# -----------------------------
# Dataset sederhana
# -----------------------------
data = [
    {"Cuaca": "Cerah",   "Suhu": "Panas",  "Kelembapan": "Tinggi", "Bermain": "Tidak"},
    {"Cuaca": "Cerah",   "Suhu": "Panas",  "Kelembapan": "Normal", "Bermain": "Tidak"},
    {"Cuaca": "Berawan", "Suhu": "Panas",  "Kelembapan": "Tinggi", "Bermain": "Ya"},
    {"Cuaca": "Hujan",   "Suhu": "Sejuk",  "Kelembapan": "Tinggi", "Bermain": "Ya"},
    {"Cuaca": "Hujan",   "Suhu": "Dingin", "Kelembapan": "Normal", "Bermain": "Ya"},
    {"Cuaca": "Hujan",   "Suhu": "Dingin", "Kelembapan": "Tinggi", "Bermain": "Tidak"},
]

features = ["Cuaca", "Suhu", "Kelembapan"]
target = "Bermain"

In [ ]:
# -----------------------------
# Fungsi bantu
# -----------------------------
def entropy(dataset, target_col):
    """Menghitung entropy dataset."""
    total = len(dataset)
    if total == 0:
        return 0

    counts = Counter(row[target_col] for row in dataset)
    ent = 0
    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)
    return ent


def partition(dataset, feature):
    """Membagi dataset berdasarkan satu fitur."""
    groups = defaultdict(list)
    for row in dataset:
        groups[row[feature]].append(row)
    return dict(groups)


def majority_label(dataset, target_col):
    """Mengambil label mayoritas."""
    counts = Counter(row[target_col] for row in dataset)
    return counts.most_common(1)[0][0]


def information_gain(dataset, feature, target_col):
    """Menghitung IG = entropy awal - remaining entropy."""
    total_entropy = entropy(dataset, target_col)
    groups = partition(dataset, feature)

    remaining_entropy = 0
    total = len(dataset)

    for subset in groups.values():
        weight = len(subset) / total
        remaining_entropy += weight * entropy(subset, target_col)

    return total_entropy - remaining_entropy

In [ ]:
# -----------------------------
# Implementasi ID3
# -----------------------------
def id3(dataset, features, target_col, parent_dataset=None):
    """
    Implementasi ID3 yang mengikuti pseudocode:
    1. Jika semua label sama -> return leaf
    2. Jika fitur habis -> return label mayoritas di D
    3. Jika D kosong -> return label mayoritas parent
    4. Pilih atribut terbaik (IG tertinggi)
    5. Buat node
    6. Partisi data
    7. Rekursif untuk tiap partisi
    """

    # Baris 3 pseudocode: jika D kosong
    if len(dataset) == 0:
        return majority_label(parent_dataset, target_col)

    labels = [row[target_col] for row in dataset]

    # Baris 1: jika semua instance punya label sama
    if len(set(labels)) == 1:
        return labels[0]

    # Baris 2: jika fitur habis
    if len(features) == 0:
        return majority_label(dataset, target_col)

    # Baris 5: pilih atribut terbaik
    best_feature = max(features, key=lambda f: information_gain(dataset, f, target_col))

    # Baris 6: buat node baru
    tree = {best_feature: {}}

    # Baris 7: partition D menggunakan best_feature
    groups = partition(dataset, best_feature)

    # Baris 8: remove best_feature from features
    remaining_features = [f for f in features if f != best_feature]

    # Baris 9-10: for each partition Di of D do ...
    for feature_value, subset in groups.items():
        subtree = id3(
            dataset=subset,
            features=remaining_features,
            target_col=target_col,
            parent_dataset=dataset
        )
        tree[best_feature][feature_value] = subtree

    return tree

In [ ]:

# -----------------------------
# Cetak pohon
# -----------------------------
def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(indent + "→ " + str(tree))
        return

    for feature, branches in tree.items():
        print(indent + str(feature))
        for value, subtree in branches.items():
            print(indent + f"├── {value}", end=" ")
            if isinstance(subtree, dict):
                print()
                print_tree(subtree, indent + "│   ")
            else:
                print(f"→ {subtree}")

In [ ]:
# -----------------------------
# Jalankan
# -----------------------------
print("Entropy awal:", round(entropy(data, target), 4))
print()

for f in features:
    print(f"IG({f}) =", round(information_gain(data, f, target), 4))

print("\nDecision Tree:")
tree = id3(data, features, target)
print_tree(tree)

Entropy awal: 1.0

IG(Cuaca) = 0.5409
IG(Suhu) = 0.2075
IG(Kelembapan) = 0.0

Decision Tree:
Cuaca
├── Cerah → Tidak
├── Berawan → Ya
├── Hujan 
│   Suhu
│   ├── Sejuk → Ya
│   ├── Dingin 
│   │   Kelembapan
│   │   ├── Normal → Ya
│   │   ├── Tinggi → Tidak


# Analisis Hasil

Hasil perhitungan menunjukkan bahwa nilai entropy awal sebesar 1.0 menandakan dataset berada dalam kondisi ketidakpastian maksimum, di mana jumlah kelas “Ya” dan “Tidak” seimbang sehingga belum dapat diambil keputusan secara langsung.

Dari perhitungan Information Gain, atribut Cuaca memiliki nilai tertinggi (0.5409), sehingga dipilih sebagai akar (root) dalam decision tree. Hal ini menunjukkan bahwa Cuaca merupakan faktor paling dominan dalam mengurangi ketidakpastian data dibandingkan atribut lainnya. Sebaliknya, atribut Kelembapan memiliki nilai Information Gain sebesar 0, yang berarti tidak memberikan kontribusi signifikan dalam pemisahan data pada tahap awal.

Struktur decision tree yang dihasilkan menunjukkan bahwa pada kondisi tertentu, keputusan dapat langsung diambil tanpa perlu analisis lanjutan, seperti pada Cuaca = Cerah (Tidak bermain) dan Cuaca = Berawan (Bermain). Namun, pada kondisi Cuaca = Hujan, proses pengambilan keputusan menjadi lebih kompleks karena masih terdapat variasi hasil, sehingga diperlukan atribut tambahan seperti Suhu dan Kelembapan untuk menentukan keputusan akhir.

Hal ini mengindikasikan bahwa beberapa kondisi memerlukan analisis yang lebih mendalam, sementara kondisi lainnya dapat diselesaikan secara langsung. Secara keseluruhan, model ini tidak hanya berfungsi untuk prediksi, tetapi juga membantu mengidentifikasi faktor-faktor utama dan pola pengambilan keputusan dalam dataset.

